# Session 1: Clinical NLP with OpenAI GPT API (via OpenRouter)

---

이 튜토리얼에서는 **OpenRouter**를 통해 OpenAI GPT 모델을 호출하여 임상노트(Clinical Note)에서 유용한 정보를 추출하는 방법을 실습합니다.

### Why OpenRouter?

- **Zero Data Retention (ZDR)**: 프롬프트 및 응답 데이터가 제공자 서버에 저장되지 않습니다.
- **`data_collection: deny`**: 제공자의 모델 학습에 데이터가 사용되지 않습니다.
- 다양한 LLM 제공자를 단일 API로 사용할 수 있습니다.

### Learning Objectives

| Task | Description |
|------|-------------|
| **Task 1** | Drug Name Extraction — 약물명 및 ATC 코드 추출 |
| **Task 2** | Adverse Drug Reaction Detection — 약물 부작용 탐지 및 검증 |
| **Task 3** | Clinical Note Summarization — 임상노트 한국어 요약 |

### Requirements

- `openai` >= 1.0
- OpenRouter API Key ([https://openrouter.ai/keys](https://openrouter.ai/keys))

---
## 0. Environment Setup

In [20]:
from openai import OpenAI
import pandas as pd
import requests
import time

# ── Configuration ──────────────────────────────────────────────
PATH = "./DATA/"
OPENROUTER_API_KEY = "" # "" 안에 openrouter api key를 복사해서 붙여넣기를 진행합니다.

In [7]:
# ── Download sample clinical notes ─────────────────────────────
BASE_URL = "https://raw.githubusercontent.com/leelabsg/clinical-llm-tutorial/main/Session1"

!wget -qP $PATH {BASE_URL}/note1_ShorthandStyle.txt
!wget -qP $PATH {BASE_URL}/note2_MIMICstyle.txt

print("Download complete.")

---
## Clinical Note Examples

본 실습에서는 두 가지 스타일의 임상노트를 제공합니다.

| Note | Style | Description |
|------|-------|-------------|
| `note1` | **Shorthand** | 간단한 요약형 노트 (약어, 목록 중심) |
| `note2` | **Narrative (MIMIC-style)** | 서술형 노트 (상세한 문맥과 진단 경과 포함) |

두 노트를 각각 사용하여 prompt 차이, 언어 차이, 노트 형식 차이에 따른 분석 결과를 비교해볼 수 있습니다.

### Note 1 — Shorthand Style

In [ ]:
with open(f"{PATH}note1_ShorthandStyle.txt", "r", encoding="utf-8") as f:
    note1 = f.read()

print(note1)

### Note 2 — Narrative Style (MIMIC-style)

In [ ]:
with open(f"{PATH}note2_MIMICstyle.txt", "r", encoding="utf-8") as f:
    note2 = f.read()

print(note2[:500])  # 앞 500자만 출력

### Select Working Note

본 실습에서는 `note1` (shorthand version)을 사용합니다.  
Narrative version을 테스트하려면 아래를 `note = note2`로 변경하세요.

In [22]:
note = note1

---
## 1. Initialize OpenRouter Client

OpenAI Python SDK의 `base_url`을 OpenRouter로 지정하여 동일한 인터페이스로 사용합니다.

**Privacy 설정:**
- `zdr: True` — Zero Data Retention: 제공자가 데이터를 저장하지 않는 엔드포인트로만 라우팅
- `data_collection: "deny"` — 프롬프트/응답이 모델 학습에 사용되지 않도록 차단

In [23]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# ── Privacy provider config (reused across all requests) ──────
PROVIDER_CONFIG = {
    "zdr": True,
    "data_collection": "deny",
}

---
## 1-1. ZDR & Provider Verification

ZDR이 정상적으로 적용되는지, 그리고 어떤 제공자(provider)를 통해 요청이 처리되었는지 확인합니다.

### (a) ZDR 지원 엔드포인트 확인

OpenRouter에서 현재 ZDR을 지원하는 모델 목록을 조회하고, 우리가 사용할 모델이 포함되어 있는지 확인합니다.

In [24]:
MODEL = "openai/gpt-5.6-sol"
VERIFIER_MODEL = "openai/gpt-5.4-mini"

zdr_response = requests.get(
    "https://openrouter.ai/api/v1/endpoints/zdr"
)
zdr_response.raise_for_status()

zdr_data = zdr_response.json()

zdr_models = {
    ep["model_id"]
    for ep in zdr_data.get("data", [])
}

for model_name in [MODEL, VERIFIER_MODEL]:
    status = (
        "ZDR supported"
        if model_name in zdr_models
        else "NOT in ZDR list"
    )
    print(f"{model_name}: {status}")

print(f"\nTotal ZDR endpoints: {len(zdr_data.get('data', []))}")

openai/gpt-5.6-sol: ZDR supported
openai/gpt-5.4-mini: ZDR supported

Total ZDR endpoints: 719


### (b) Test API Call & Provider 확인

간단한 테스트 요청을 ZDR 옵션과 함께 전송하고, 응답에서 generation ID를 확인합니다.

이 generation ID를 OpenRouter의 Activity / Logs에서 조회하면 실제 요청을 처리한 provider(Azure, Bedrock 등)를 확인할 수 있습니다.   
이를 통해 ZDR routing 조건이 적용된 요청이 어떤 provider로 전달되었는지 검증할 수 있습니다.

In [27]:
# Test request with ZDR enabled
test_response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'ZDR test OK' and nothing else."}],
    extra_body={"provider": PROVIDER_CONFIG},
)

print("Response      :", test_response.choices[0].message.content)
print("Model         :", test_response.model)
print("Generation ID :", test_response.id)

Response      : ZDR test OK
Model         : openai/gpt-5.6-sol
Generation ID : gen-1786420654-dtlpYlEqPIgR5V3XsL49


---
## Task 1: Drug Name Extraction & Drug Code Identification

임상노트로부터 언급된 약물명을 추출하는 작업을 수행합니다.

- GPT에게 임상노트를 입력하고, 텍스트에서 **약물명** 추출 및 약물에 상응하는 **ATC CODE**를 추출합니다.
- 약물은 **일반명** 또는 **약어**(예: MFM = metformin)로 표현될 수 있습니다.

### 1-1. Drug Name Only

In [28]:
prompt1_eng = f"""
Task: Extract all drug names mentioned in the following clinical note.
- Drug names may be written as abbreviations.
- Return the list of drug names only, without any additional explanation.

Clinical Note:
{note}
"""

In [29]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a clinical text extraction assistant."},
        {"role": "user", "content": prompt1_eng},
    ],
    extra_body={"provider": PROVIDER_CONFIG},
)

print(response.choices[0].message.content)

Metformin
Levemir
Insulin
Losartan
Atorvastatin


### 1-2. Drug Name + Description + ATC Code

In [30]:
prompt2_eng = f"""
Task: Extract all drug names mentioned in the following clinical note.
- Drug names may be written as abbreviations.
- Return the list of drug names, simple descriptions, and corresponding ATC codes in a table format (CSV).

Clinical Note:
{note}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a clinical text extraction assistant."},
        {"role": "user", "content": prompt2_eng},
    ],
    extra_body={"provider": PROVIDER_CONFIG},
)

print(response.choices[0].message.content)

Drug name,Simple description,ATC code
"Metformin (MTF)","Biguanide oral antidiabetic; discontinued due to suspected lactic acidosis","A10BA02"
"Insulin detemir (Levemir)","Long-acting basal insulin analogue","A10AE05"
"Losartan","Angiotensin II receptor blocker used for hypertension","C09CA01"
"Atorvastatin","HMG-CoA reductase inhibitor used for dyslipidemia","C10AA05"


### 1-3. Usage Check

사용한 토큰 수를 확인할 수 있습니다.

In [31]:
print(response.usage)

CompletionUsage(completion_tokens=315, prompt_tokens=251, total_tokens=566, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=201, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_tokens=0), cost=0.010705, is_byok=False, cost_details={'upstream_inference_cost': 0.010705, 'upstream_inference_prompt_cost': 0.001255, 'upstream_inference_completions_cost': 0.00945})


---
## Task 2: Adverse Drug Reaction (ADR) Detection

임상노트에 언급된 **약물 부작용(Adverse Drug Reactions, ADR)** 을 탐지하는 작업을 수행합니다.

- GPT에게 임상노트를 입력하고, 특정 약물 투여 이후 발생한 부작용을 **약물-증상 쌍(drug → symptom)** 형태로 추출합니다.
- 약물명은 약어로 표현될 수 있으며, 부작용은 명시적 또는 암시적으로 표현되어 있을 수 있습니다.

### 2-1. ADR Extraction

In [32]:
prompt_adr = f"""
Task: Extract all possible adverse drug reactions (ADRs) mentioned in the clinical note.
- Return results in JSON format, with each item containing:
  - "drug": drug name
  - "symptom": symptom or adverse effect
  - "evidence": supporting sentence or phrase from the note
- No extra explanation.

Clinical Note:
{note}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a clinical text extraction assistant."},
        {"role": "user", "content": prompt_adr},
    ],
    extra_body={"provider": PROVIDER_CONFIG},
)

print(response.choices[0].message.content)

[
  {
    "drug": "Metformin (MTF)",
    "symptom": "Diarrhea",
    "evidence": "MTF → 설사"
  },
  {
    "drug": "Metformin (MTF)",
    "symptom": "Abdominal bloating",
    "evidence": "MTF → 복부팽만"
  },
  {
    "drug": "Metformin (MTF)",
    "symptom": "Decreased appetite",
    "evidence": "MTF → 식욕저하"
  },
  {
    "drug": "Metformin (MTF)",
    "symptom": "Lethargy",
    "evidence": "MTF → 무기력감"
  },
  {
    "drug": "Metformin (MTF)",
    "symptom": "Suspected lactic acidosis",
    "evidence": "MTF-induced lactic acidosis 의심 → 약 중단 후 회복"
  }
]


### 2-2. Verification Agent

LLM이 생성한 ADR 추출 결과가 타당한지 검증하기 위해, **별도의 Verifier Agent**를 생성하여 답변을 교차 검증합니다.

In [33]:
verifier_prompt = """
If you disagree, provide counterarguments.
Provide agree and comment for each drug and side effect pairs.
Respond in JSON:
{"agree": true/false, "comments": "..."}.
"""

extractor_output = response.choices[0].message.content

verifier_input = (
    f"{verifier_prompt}\n\n"
    f"ExtractorAgent output:\n{extractor_output}\n\n"
    f"Medical note:\n{note}"
)

verifier_response = client.chat.completions.create(
    model=VERIFIER_MODEL,  # lightweight model for verification
    messages=[
        {"role": "system", "content": "You are VerifierAgent. Your job is to check ExtractorAgent's identified side effects."},
        {"role": "user", "content": verifier_input},
    ],
    extra_body={"provider": PROVIDER_CONFIG},
)

print(verifier_response.choices[0].message.content)

{"agree": true, "comments": "1) Metformin (MTF) → Diarrhea: agree. The note explicitly states 'MTF → 설사'. 2) Metformin (MTF) → Abdominal bloating: agree. Explicitly supported by 'MTF → 복부팽만'. 3) Metformin (MTF) → Decreased appetite: agree. Explicitly supported by 'MTF → 식욕저하'. 4) Metformin (MTF) → Lethargy: agree. Explicitly supported by 'MTF → 무기력감'. 5) Metformin (MTF) → Suspected lactic acidosis: agree. The note says 'Lactic acidosis (Lac 3.9) 의심' and 'MTF-induced lactic acidosis 의심 → 약 중단 후 회복', which directly supports this adverse effect."}


---
## Task 3: Clinical Note Summarization

임상노트에 포함된 핵심 정보를 GPT를 통해 한글로 **요약(summarization)** 하는 작업을 수행합니다.

- 임상노트에는 환자의 병력, 진단, 투약, 검사 결과, 치료 계획 등의 정보가 포함되어 있으며, 종종 매우 길고 복잡합니다.
- 이 Task는 환자에 대한 중요한 정보를 요약하여, 환자/보호자 설명에 활용될 수 있는 간결한 요약문을 생성하는 것이 목적입니다.
- 출력은 한국어 요약문으로 생성됩니다.

In [34]:
prompt_family = f"""
Task: Summarize the following clinical note in simple language for the patient's family.
- Use non-technical, plain Korean language.
- The summary should explain what happened, what was diagnosed, how it was treated, and what to expect.
- Avoid complex terminology and abbreviations.

Output language: Korean

Clinical Note:
{note2}
"""

In [35]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a clinical text summarization assistant."},
        {"role": "user", "content": prompt_family},
    ],
    extra_body={"provider": PROVIDER_CONFIG},
)

print(response.choices[0].message.content)

### 입원 경과 요약

- 환자분은 78세 여성으로, 당뇨병·고혈압·만성 콩팥병이 있습니다.
- 약 2주 전 당뇨 조절을 위해 **메트포르민**을 시작한 뒤 설사와 배 아픔이 계속되었고, 식사량과 기운이 줄었습니다.
- 입원 당시 탈수로 혈압이 매우 낮았고 맥박이 빨랐습니다. 콩팥 기능도 평소보다 나빠졌으며, 몸에 산성 물질이 쌓여 혈액이 산성으로 변한 상태가 확인되었습니다.
- 의료진은 콩팥 기능이 좋지 않은 상태에서 메트포르민을 복용하면서 생긴 부작용으로 판단했습니다.

### 치료와 결과

- 메트포르민을 즉시 중단했습니다.
- 정맥으로 수액과 혈액의 산성 상태를 교정하는 약을 투여했습니다.
- 48시간 안에 혈압과 의식 상태가 좋아졌고, 콩팥 기능도 수액 치료 후 호전되었습니다.
- 투석은 필요하지 않았습니다.
- 검사상 폐렴이나 요로 감염의 증거는 없었습니다.

### 퇴원 후 관리

- **메트포르민은 앞으로 절대 다시 복용하면 안 됩니다.**
- 당뇨병은 매일 밤 지속형 인슐린 6단위로 조절할 예정입니다.
- 혈당은 하루 두 번 확인하고, 결과에 따라 의료진이 인슐린 용량을 조절해야 합니다.
- 그 외 혈압약, 콜레스테롤약, 위장약 등은 처방대로 복용합니다.
- 요양시설로 돌아가며, 1~2주 안에 콩팥 및 당뇨 전문 진료를 받아야 합니다.

설사, 식사 거부, 심한 기운 없음, 어지럼증, 의식 변화, 소변량 감소가 다시 나타나면 즉시 의료진에게 알려야 합니다.


---

## Summary

| Task | Method | Key Takeaway |
|------|--------|------|
| **Drug Extraction** | Prompt engineering | 약어 포함 약물명 + ATC 코드 추출 가능 |
| **ADR Detection** | Extractor + Verifier | Multi-agent 패턴으로 결과 신뢰도 향상 |
| **Summarization** | Korean plain-language | 환자/보호자용 비전문 요약 생성 |

### OpenRouter Privacy Settings

| Parameter | Value | Effect |
|-----------|-------|--------|
| `zdr` | `True` | ZDR 정책을 준수하는 엔드포인트로만 라우팅 |
| `data_collection` | `"deny"` | 프롬프트/응답이 모델 학습에 사용되지 않음 |
